In [4]:
!pip install sentencepiece

In [5]:
import psutil # for ram
import torch
import os
import random
import numpy as np
import torch.nn as nn
#import tiktoken as tk

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
import sentencepiece as spm
# Train the model
spm.SentencePieceTrainer.train(
    input='training-data-expanded_3.txt',       # Your training data
    model_prefix='misovic',    # Output model name
    vocab_size=10000,          # Desired vocabulary size
    model_type='unigram',     # Algorithm: 'unigram' or 'bpe'
    character_coverage=1.0    # 1.0 for English, 0.9995 for CJK
)

tk = spm.SentencePieceProcessor(model_file='misovic.model') #spm

with open("training-data-expanded_3.txt", "r", encoding="utf-8") as f:
    data = f.read()
token_ids = tk.encode(data, out_type=int) # convert to manual char level tokenization - token_ids, where data is the training corpus

In [7]:
#INPUTS, tokenization and embedding
vocab_size = 10000
embed_dim = 256
batch_size = 20
block_size = 64 # how big is the context window for X and Y, where Y is X shifted to the right, aka the next element after X
num_heads = 4
num_layers = 6 # how many blocks of attention/ff

In [8]:
#using the architechture to construct the lm

class lm(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, block_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim) # return the token embed_dim sized vector
        self.pos_emb = nn.Embedding(block_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True
        )
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(embed_dim, vocab_size) # return logits for the whole vocab

    def forward(self, batch):
        B,T = batch.shape #20, 64-batch length
        pos = torch.arange(T,device=batch.device)
        x = self.embedding(batch) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=batch.device)
        x = self.blocks(x, mask=mask, is_causal=True) # namesti encoder da bude kao decoder
        logits = self.lm_head(x)

        return logits

model = lm(vocab_size, embed_dim, num_heads, num_layers, block_size)
model = model.to(device)

In [9]:
#load the previous model
if os.path.exists("model_checkpoint.pt"):
    model.load_state_dict(torch.load("model_checkpoint.pt", map_location=device))
    print("Resumed from checkpoint.")

In [10]:
#training the model - making the dataset class

class LanguageModelDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        x = torch.tensor(self.token_ids[idx : idx + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[idx + 1 : idx + self.block_size + 1], dtype=torch.long)
        return x, y

from torch.utils.data import DataLoader

N = len(token_ids)
chunk=10000
chunks = [token_ids[i:i+chunk] for i in range(0,N,chunk)]
random.seed(42)
random.shuffle(chunks)
token_ids = [t for ch in chunks for t in ch]
train_ids = token_ids[:int(0.8 * N)]
val_ids   = token_ids[int(0.8 * N):int(0.9 * N)]
test_ids  = token_ids[int(0.9 * N):]

#training data is for train_ids
dataset = LanguageModelDataset(train_ids, block_size)
dataset_val = LanguageModelDataset(val_ids, block_size)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle=True, drop_last=True)
val_dataloader = DataLoader(dataset_val, batch_size = batch_size, shuffle=True, drop_last=True)

import torch.optim as optim

# Adam optimizer & epoch count:
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
num_epochs = 2


loss_fn = nn.CrossEntropyLoss()
try:
    for epoch in range(num_epochs):
        c = 0
        total_loss = 0
        for batch_X, batch_Y in dataloader:
            #if c >= 25000: break
            if c % 150000 == 0 and c!=0:
              for param_group in optimizer.param_groups:
                  param_group['lr'] *= 0.33
              print("Learning rate lowered!")
            #move from the cpu to gpu
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)
            #forward pass
            logits = model(batch_X) # should be batch_size, block_size, vocab_size
            #print(logits.shape)
            # compute loss
            loss = loss_fn(logits.view(-1, vocab_size), batch_Y.view(-1))
            optimizer.zero_grad()  # clear old gradients
            loss.backward()        # backprop / compute gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()       # update weights
            total_loss += loss.item()
            c+=1
            if c % 100 == 0 and psutil.virtual_memory().percent > 90:
                print("RAM is gettin filled up too much, stopping and saving...")
                torch.save(model.state_dict(), "model_checkpoint.pt")
                break
            if c % 500 == 0:
                print(f"Iteration: {c}, loss: {loss.item():.4f}")
            if c % 5000 == 0:
                model.eval()
                with torch.no_grad():
                    vX, vY = next(iter(val_dataloader))
                    v_logits = model(vX.to(device))
                    v_loss = loss_fn(v_logits.view(-1, vocab_size), vY.to(device).view(-1))
                model.train()
                print(f"iter {c}: train {loss.item():.3f} | val {v_loss.item():.3f}")
        torch.save(model.state_dict(), "model_checkpoint.pt")
        for param_group in optimizer.param_groups:
            param_group['lr'] *= 0.33
        print("Learning rate lowered!")
except KeyboardInterrupt:
    print("Training interrupted. Saving model...")
    torch.save(model.state_dict(), "model_checkpoint.pt")
    print("Model saved.")

Iteration: 500, loss: 6.2815
Iteration: 1000, loss: 5.8536
Iteration: 1500, loss: 5.8112
Iteration: 2000, loss: 5.8361
Iteration: 2500, loss: 5.7036
Iteration: 3000, loss: 5.1482
Iteration: 3500, loss: 5.2964
Iteration: 4000, loss: 5.2533
Iteration: 4500, loss: 4.9952
Iteration: 5000, loss: 5.4873
iter 5000: train 5.487 | val 4.945
Iteration: 5500, loss: 4.9130
Iteration: 6000, loss: 5.4422
Iteration: 6500, loss: 4.8106
Iteration: 7000, loss: 5.1091
Iteration: 7500, loss: 4.7893
Iteration: 8000, loss: 4.6518
Iteration: 8500, loss: 5.0383
Iteration: 9000, loss: 4.5277
Iteration: 9500, loss: 4.9263
Iteration: 10000, loss: 4.9650
iter 10000: train 4.965 | val 4.713
Iteration: 10500, loss: 4.8382
Iteration: 11000, loss: 4.6006
Iteration: 11500, loss: 4.7316
Iteration: 12000, loss: 4.7361
Iteration: 12500, loss: 4.5896
Iteration: 13000, loss: 4.7502
Iteration: 13500, loss: 4.7803
Iteration: 14000, loss: 4.6800
Iteration: 14500, loss: 4.8735
Iteration: 15000, loss: 4.5949
iter 15000: train 4

In [11]:
#outputing the next token
prompt = input("Enter your prompt: ")
#logits = model(encoded_prompt)   →   (1, T, 10000)
#lm_head = (10000, 256)
def generate(prompt, max_tokens, temperature=0.7):
    model.eval() # no dropout anymore
    ids = tk.encode(prompt, out_type=int)
    encoded_prompt = torch.tensor([ids], dtype=torch.long, device=device) # [] to make it 2D
    with torch.no_grad():
        for _ in range(max_tokens):
            context = encoded_prompt[:, -block_size:] #model limitted to block_size, take the last block_size amount of tokens
            logits = model(context) #id's of the prompt, this gives us the next token
            last = logits[:, -1, :] # token id is the index of the matrix itself
            #[dim0:dim1:dim2], dim0 - batch, dim1 - token id out of them all, dim2- vocab
            probs = torch.softmax(last / temperature, dim=-1) #-1, ditch the other dimension, last is shape(1,10000) no T since we -1'd it in logits
            next_token = torch.multinomial(probs, num_samples=1) #multinomial picks 40% chance 40% of the time

            encoded_prompt = torch.cat([encoded_prompt, next_token], dim=1)
        decoded_prompt = tk.decode(encoded_prompt[0].tolist()) #back to text
    return decoded_prompt

generate(prompt, 200) #run the model

Enter your prompt: Hello said the


"Hello said the mighty Host; and then the King Shall be the mightiest left. Hee ended, and th' Ethereal mould Interpretain, who shall find Faith High Eternal Justice ordain'd, and one Law all his right. Thus was the Heav'n, his Name found, Or Earth, when the Sun sov'n, over Night Equal from the Earth, Adversarie, each in thir great Conquerour, and in Hell Thir JAPHIAN Cover'd, describing the pleasant Sea, and from the Lake This Rivers translated descended on the op'ning Gulf, Which now he now beheld Or Earth, as from the Pest Therefore he appeers, And ever since created a calm, obscure, and in one Aire The Sons of God, which in his name be given To all mankind, hath not in the world, nor incurs'd Not of his happie thoughts. Back to him"

In [12]:
#print parameters of model & tokens of dataset
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Dataset tokens: {len(dataset):,}")

Model parameters: 9,884,944
Dataset tokens: 2,493,548
